# Customer Intelligence & Churn Prediction Platform

## Phase 4: Feature Engineering

**Objective**

Create new business features from the cleaned datasets to support customer analysis, segmentation, and churn prediction.

**Dataset**

Olist Brazilian E-Commerce Public Dataset (Cleaned)

**Author**

Tanish Mhatre

---

## Notebook Roadmap

**1. Load Cleaned Datasets**
- Load processed datasets.

**2. Create Order Features**
- Create delivery and order time features.

**3. Create Customer Features**
- Calculate customer purchase metrics.

**4. Create Payment Features**
- Generate payment-related features.

**5. Create Product Features**
- Create product-level metrics.

**6. Save Feature Dataset**
- Save datasets with new features.

---

In [1]:
# Import library
import pandas as pd

# Load cleaned datasets

customers = pd.read_csv("../data/processed/customers_clean.csv")
orders = pd.read_csv("../data/processed/orders_clean.csv")
order_items = pd.read_csv("../data/processed/order_items_clean.csv")
payments = pd.read_csv("../data/processed/payments_clean.csv")
reviews = pd.read_csv("../data/processed/reviews_clean.csv")
products = pd.read_csv("../data/processed/products_clean.csv")
sellers = pd.read_csv("../data/processed/sellers_clean.csv")

# Store datasets together

datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers
}

# Check dataset size

for name, df in datasets.items():
    print(name, df.shape)

customers (99441, 5)
orders (99441, 8)
order_items (112650, 7)
payments (103886, 5)
reviews (99224, 7)
products (32951, 9)
sellers (3095, 4)


### Prepare Date Columns

Convert date columns to datetime format before creating new features.

In [6]:
# Order date columns

order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

# Convert to datetime

for col in order_date_columns:
    orders[col] = pd.to_datetime(orders[col])

print(orders[order_date_columns].dtypes)

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


**Create Order Features**

Create new features from order dates to measure delivery performance.

### Delivery Days

Calculate the number of days taken to deliver each order.

In [7]:
# Create delivery days feature

orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.days

In [9]:
orders[
    ["order_purchase_timestamp",
        "order_delivered_customer_date",
        "delivery_days"]].head()

,order_purchase_timestamp,order_delivered_customer_date,delivery_days
0,2017-10-02 10:56:33,2017-10-10 21:25:13,8.0
1,2018-07-24 20:41:37,2018-08-07 15:27:45,13.0
2,2018-08-08 08:38:49,2018-08-17 18:06:29,9.0
3,2017-11-18 19:28:06,2017-12-02 00:28:42,13.0
4,2018-02-13 21:18:39,2018-02-16 18:17:02,2.0


### Approval Time

Calculate the time taken to approve each order after purchase.

In [10]:
# Create approval time feature (in hours)

orders["approval_time_hours"] = (
    orders["order_approved_at"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / 3600

In [11]:
orders[
    [
        "order_purchase_timestamp",
        "order_approved_at",
        "approval_time_hours"
    ]
].head()

,order_purchase_timestamp,order_approved_at,approval_time_hours
0,2017-10-02 10:56:33,2017-10-02 11:07:15,0.178333
1,2018-07-24 20:41:37,2018-07-26 03:24:27,30.713889
2,2018-08-08 08:38:49,2018-08-08 08:55:23,0.276111
3,2017-11-18 19:28:06,2017-11-18 19:45:59,0.298056
4,2018-02-13 21:18:39,2018-02-13 22:20:29,1.030556


In [12]:
orders["approval_time_hours"].describe()

count    99281.000000
mean        10.419094
std         26.038004
min          0.000000
25%          0.215000
50%          0.343333
75%         14.580833
max       4509.180556
Name: approval_time_hours, dtype: float64

### Late Delivery

Identify whether orders were delivered after the estimated delivery date.

In [16]:
# Create late delivery feature

orders["late_delivery"] = (
    orders["order_delivered_customer_date"]
    > orders["order_estimated_delivery_date"]
)
orders[
    [
        "order_estimated_delivery_date",
        "order_delivered_customer_date",
        "late_delivery"
    ]
].head()



,order_estimated_delivery_date,order_delivered_customer_date,late_delivery
0,2017-10-18,2017-10-10 21:25:13,False
1,2018-08-13,2018-08-07 15:27:45,False
2,2018-09-04,2018-08-17 18:06:29,False
3,2017-12-15,2017-12-02 00:28:42,False
4,2018-02-26,2018-02-16 18:17:02,False


In [17]:
orders["late_delivery"].value_counts()

late_delivery
False    91614
True      7827
Name: count, dtype: int64

### Delivery Difference

Calculate the difference between estimated and actual delivery dates.


In [21]:
# Create delivery difference feature

orders["delivery_difference_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.days

orders[
    [
        "order_estimated_delivery_date",
        "order_delivered_customer_date",
        "delivery_difference_days"
    ]
].head()



,order_estimated_delivery_date,order_delivered_customer_date,delivery_difference_days
0,2017-10-18,2017-10-10 21:25:13,-8.0
1,2018-08-13,2018-08-07 15:27:45,-6.0
2,2018-09-04,2018-08-17 18:06:29,-18.0
3,2017-12-15,2017-12-02 00:28:42,-13.0
4,2018-02-26,2018-02-16 18:17:02,-10.0


In [20]:
orders["delivery_difference_days"].describe()

count    96476.000000
mean       -11.876881
std         10.183854
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
Name: delivery_difference_days, dtype: float64

### Customer Order Frequency

Calculate the total number of orders placed by each customer.

In [23]:
# Count orders per customer

customer_frequency = (
    orders
    .groupby("customer_id")
    .size()
    .reset_index(name="total_orders")
)

customer_frequency.head()

,customer_id,total_orders
0,00012a2ce6f8dcda20d059ce98491703,1
1,000161a058600d5901f007fab4c27140,1
2,0001fd6190edaaf884bcaf3d49edf079,1
3,0002414f95344307404f0ace7a26f1d5,1
4,000379cdec625522490c315e70c7a9fb,1


### Customer Identification

Use customer_unique_id to identify actual customers because one customer can have multiple orders.

In [26]:
customers[
    [
        "customer_id",
        "customer_unique_id"
    ]
].head()

,customer_id,customer_unique_id
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066


In [25]:
customers[
    [
        "customer_id",
        "customer_unique_id"
    ]
].head()

,customer_id,customer_unique_id
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066


In [28]:
# Add real customer ID to orders

orders_customer = orders.merge(
    customers[
        [
            "customer_id",
            "customer_unique_id"
        ]
    ],
    on="customer_id",
    how="left"
)
orders_customer[
    [
        "order_id",
        "customer_id",
        "customer_unique_id"
    ]
].head()

,order_id,customer_id,customer_unique_id
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6


In [33]:
# Count orders per real customer

customer_frequency = (
    orders_customer
    .groupby("customer_unique_id")
    .size().reset_index(name="total_orders")
)
customer_frequency.head()

,customer_unique_id,total_orders
0,0000366f3b9a7992bf8c76cfdf3221e2,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1
2,0000f46a3911fa3c0805444483337064,1
3,0000f6ccb0745a6a4b88665a16c9f078,1
4,0004aac84e0df4da2b147fca70cf8255,1


In [32]:
customer_frequency["total_orders"].value_counts()

total_orders
1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64

### Customer Monetary Value

Calculate the total amount spent by each customer.

In [34]:
# Create item value

order_items["item_value"] = (
    order_items["price"] + order_items["freight_value"]
)
# Calculate total order value

order_value = (
    order_items
    .groupby("order_id")["item_value"].sum().reset_index(name="order_value")
)
order_value.head()


,order_id,order_value
0,00010242fe8c5a6d1ba2dd792cb16214,72.19
1,00018f77f2f0320c557190d7a144bdd3,259.83
2,000229ec398224ef6ca0657da4fc703e,216.87
3,00024acbcdf0a6daa1e931b038114c75,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04


### Merge Order Value with Customer Orders

Combine order values with customer information to calculate customer spending.

In [37]:
# Merge order value with customer orders

orders_customer = orders_customer.merge(
    order_value,
    on="order_id",
    how="left"
)
orders_customer[
    [
        "order_id",
        "customer_unique_id",
        "order_value"
    ]
].head()

,order_id,customer_unique_id,order_value
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,af07308b275d755c9edb36a90c618231,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,3a653a41f6f9fc3d2a113cf8398680e8,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,7c142cf63193a1473d2e66489a9ae977,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,72632f0f9dd73dfee390c9b22eb56dd6,28.62


### Customer Monetary Value

Calculate the total amount spent by each customer.

In [39]:
# Calculate total spending per customer

customer_monetary = (
    orders_customer.groupby("customer_unique_id")["order_value"].sum()
    .reset_index(name="total_spent")
)

customer_monetary.head()


,customer_unique_id,total_spent
0,0000366f3b9a7992bf8c76cfdf3221e2,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,27.19
2,0000f46a3911fa3c0805444483337064,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,43.62
4,0004aac84e0df4da2b147fca70cf8255,196.89


In [40]:
customer_monetary["total_spent"].describe()

count    96096.000000
mean       164.872141
std        227.938658
min          0.000000
25%         62.390000
50%        107.270000
75%        182.237500
max      13664.080000
Name: total_spent, dtype: float64

### Customer Spending Analysis

Analyze the distribution of total spending for each customer.

In [41]:
customer_monetary.sort_values(
    by="total_spent",
    ascending=False
).head(10)

,customer_unique_id,total_spent
3826,0a0a92112bd4c708ca5fde585afaa872,13664.08
81962,da122df9eeddfedc1dc1f5349a1a690c,7571.63
44447,763c8b1c9c68a0229c42c9fc6f662b93,7274.88
82808,dc4802a71eae9be1dd28f5d788ceb526,6929.31
26205,459bef486812aa25204be022145caa62,6922.21
95806,ff4159b92c40ebe40454e3e6a7c35ed6,6726.66
24121,4007669dec559734d6f53e029e360987,6081.54
35070,5d0a2980b292d049061542014e8960bf,4809.44
89688,eebb5dda148d3893cdaf5b5ca3040ccb,4764.34
27441,48e1ac109decbb87765a3eade6854098,4681.78


### Average Order Value

Calculate the average amount spent per order for each customer.

In [42]:
customer_features = customer_frequency.merge(
    customer_monetary,
    on="customer_unique_id",
    how="left"
)

customer_features["average_order_value"] = (
    customer_features["total_spent"] /
    customer_features["total_orders"]
)

customer_features.head()

,customer_unique_id,total_orders,total_spent,average_order_value
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89


### Average Order Value

Calculate the average amount spent per order for each customer.

In [43]:
# Calculate average order value

customer_features["average_order_value"] = (
    customer_features["total_spent"] /
    customer_features["total_orders"]
)
customer_features[
    [
        "customer_unique_id",
        "total_orders",
        "total_spent",
        "average_order_value"
    ]
].head()

,customer_unique_id,total_orders,total_spent,average_order_value
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89


### Customer Recency

Find the most recent purchase date for each customer.


In [44]:
# Get the last purchase date for each customer

customer_recency = (
    orders_customer
    .groupby("customer_unique_id")["order_purchase_timestamp"]
    .max()
    .reset_index(name="last_purchase_date")
)

customer_recency.head()

,customer_unique_id,last_purchase_date
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42


### Calculate Recency

Use the latest purchase date in the dataset as the reference date.

In [45]:
# Find the latest purchase date in the dataset

reference_date = orders["order_purchase_timestamp"].max()

print(reference_date)

2018-10-17 17:30:18


### Calculate Customer Recency

Calculate the number of days between the customer's last purchase and the latest purchase in the dataset.

In [46]:
# Calculate recency in days

customer_recency["recency_days"] = (
    reference_date -
    customer_recency["last_purchase_date"]
).dt.days
customer_recency.head()

,customer_unique_id,last_purchase_date,recency_days
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,160
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,163
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,585
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,369
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,336


### Merge Customer Recency

Add the recency feature to the customer feature table.

In [47]:
# Merge recency into customer features

customer_features = customer_features.merge(
    customer_recency[[
            "customer_unique_id",
            "recency_days"]],
    on="customer_unique_id",
    how="left"
)
customer_features.head()

,customer_unique_id,total_orders,total_spent,average_order_value,recency_days
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90,160
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19,163
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22,585
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62,369
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89,336


## Validate Customer Features

Check the structure, missing values, and summary statistics of the final customer feature table.

In [48]:
customer_features.shape

(96096, 5)

In [49]:
customer_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 96096 entries, 0 to 96095
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   customer_unique_id   96096 non-null  str    
 1   total_orders         96096 non-null  int64  
 2   total_spent          96096 non-null  float64
 3   average_order_value  96096 non-null  float64
 4   recency_days         96096 non-null  int64  
dtypes: float64(2), int64(2), str(1)
memory usage: 3.7 MB


In [50]:
customer_features.isnull().sum()

customer_unique_id     0
total_orders           0
total_spent            0
average_order_value    0
recency_days           0
dtype: int64

In [51]:
customer_features.describe()

,total_orders,total_spent,average_order_value,recency_days
count,96096.000000,96096.000000,96096.000000,96096.000000
mean,1.034809,164.872141,159.810187,287.735691
std,0.214384,227.938658,220.564409,153.414676
min,1.000000,0.000000,0.000000,0.000000
25%,1.000000,62.390000,61.690000,163.000000
50%,1.000000,107.270000,105.180000,268.000000
75%,1.000000,182.237500,176.230000,397.000000
max,17.000000,13664.080000,13664.080000,772.000000


In [52]:
customer_features.sort_values(
    "total_spent",
    ascending=False
).head(10)

,customer_unique_id,total_orders,total_spent,average_order_value,recency_days
3826,0a0a92112bd4c708ca5fde585afaa872,1,13664.08,13664.080,383
81962,da122df9eeddfedc1dc1f5349a1a690c,2,7571.63,3785.815,564
44447,763c8b1c9c68a0229c42c9fc6f662b93,1,7274.88,7274.880,94
82808,dc4802a71eae9be1dd28f5d788ceb526,1,6929.31,6929.310,611
26205,459bef486812aa25204be022145caa62,1,6922.21,6922.210,83
95806,ff4159b92c40ebe40454e3e6a7c35ed6,1,6726.66,6726.660,510
24121,4007669dec559734d6f53e029e360987,1,6081.54,6081.540,327
35070,5d0a2980b292d049061542014e8960bf,1,4809.44,4809.440,97
89688,eebb5dda148d3893cdaf5b5ca3040ccb,1,4764.34,4764.340,546
27441,48e1ac109decbb87765a3eade6854098,1,4681.78,4681.780,117


In [53]:
# Save customer features for Exploratory Data Analysis

customer_features.to_csv(
    "../data/processed/customer_features.csv",
    index=False
)

print("✅ Customer features saved successfully.")

✅ Customer features saved successfully.


In [54]:
# Save order value dataset

order_value.to_csv(
    "../data/processed/order_value.csv",
    index=False
)

print("✅ order_value.csv saved successfully.")

✅ order_value.csv saved successfully.


In [55]:
import os

print(os.listdir("../data/processed"))

['customers_clean.csv', 'customer_features.csv', 'orders_clean.csv', 'order_items_clean.csv', 'order_value.csv', 'payments_clean.csv', 'products_clean.csv', 'reviews_clean.csv', 'sellers_clean.csv']


In [56]:
orders.to_csv(
    "../data/processed/orders_features.csv",
    index=False
)